# Feature Attribution using PageRank

In [1]:
import sys

!{sys.executable} -m pip install nbimporter

In [2]:
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter
import sklearn.ensemble

import Ft_Att_Rank as far
import GraphPR as gpr
import PageRank as pr

In [3]:
# https://www.kaggle.com/c/titanic
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [4]:
X_train= far.pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')
X_train= far.pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train= far.normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= far.normalize_selected_cols(X_train_ohe, numeric_columns)

X_train.shape

(891, 7)

In [5]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

In [6]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [7]:
# ML model setup

rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

In [8]:
mean_acc_all, acc_no_i, acc_no_ij= far.kfoldnn_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [9]:
print(mean_acc_all)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.855944055944056
-----------------------------
[0.88  0.866 0.859 0.376 0.871 0.857 0.871 0.856 0.856 0.867 0.867 0.871]
-----------------------------
[[0.    0.883 0.836 0.874 0.88  0.877 0.877 0.88  0.878 0.88  0.876 0.877]
 [0.878 0.    0.878 0.436 0.866 0.864 0.878 0.864 0.86  0.862 0.876 0.876]
 [0.838 0.871 0.    0.467 0.856 0.871 0.874 0.859 0.857 0.848 0.86  0.874]
 [0.87  0.436 0.46  0.    0.379 0.471 0.473 0.368 0.464 0.386 0.494 0.501]
 [0.877 0.864 0.856 0.378 0.    0.859 0.883 0.863 0.859 0.871 0.88  0.874]
 [0.878 0.862 0.863 0.476 0.864 0.    0.873 0.862 0.873 0.86  0.871 0.871]
 [0.877 0.874 0.871 0.42  0.883 0.873 0.    0.871 0.878 0.87  0.878 0.877]
 [0.878 0.878 0.862 0.383 0.876 0.876 0.878 0.    0.663 0.852 0.883 0.877]
 [0.877 0.877 0.874 0.376 0.863 0.862 0.863 0.669 0.    0.87  0.883 0.876]
 [0.877 0.871 0.87  0.378 0.855 0.857 0.871 0.869 0.869 0.    0.857 0.782]
 [0.876 0.866 0.856 0.492 0.878 0.874 0.877 0.878 0.867 0.874 0.    0.876]
 [0.878 0.876 0.874 0.4

In [11]:
num_fts= acc_no_ij.shape[0]

# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, mean_acc_all, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, mean_acc_all, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

In [51]:
import networkx as nx

def run_lib_pr(graph_matrix, ft_names):
    
    num_fts= graph_matrix.shape[0]
    
    D= nx.DiGraph()

    for i in range(num_fts):
        for j in range(num_fts):
            if (i!= j):
                D.add_weighted_edges_from([(ft_names[i],ft_names[j],graph_matrix[i,j])])
                
    pRank= pd.Series(nx.pagerank(D))

    return pRank.sort_values(ascending=False)

In [52]:
run_lib_pr(st_matrix1, X_train_ohe.columns)

Fare          0.354597
Age           0.149903
Parch         0.122303
Embarked_S    0.060973
Embarked_Q    0.056398
Pclass_3      0.050864
Pclass_2      0.047944
Sex_male      0.046006
SibSp         0.039120
Embarked_C    0.027620
Sex_female    0.026221
Pclass_1      0.018051
dtype: float64

In [49]:
# using my PR to compare results
def run_my_pr(file_path, graph_matrix, ft_names):
    
    num_fts= graph_matrix.shape[0]

    f= open(file_path, 'w')

    for i in range(num_fts):
        for j in range(num_fts):
            line= (str(i) + ',' + str(j) + ',' + str(graph_matrix[i,j]) + '\n')
            f.write(line)

    f.close()

    graph= gpr.init_graph(file_path)

    myPRank= pd.Series(pr.run_PageRank(graph, iteration= 100, damping_factor= 0.95, tolerance= 1.0e-6))
    myPRank= myPRank.sort_values(ascending=False)

    ids_names= ft_names[myPRank.index]

    myPRank.index= ids_names

    return myPRank

In [50]:
run_my_pr('datasets/FAR_data.txt', st_matrix1, X_train_ohe.columns)

Fare          0.304923
Age           0.160533
Parch         0.074396
SibSp         0.062853
Sex_female    0.054675
Embarked_C    0.051053
Pclass_3      0.049807
Pclass_2      0.049655
Pclass_1      0.048714
Embarked_S    0.048093
Embarked_Q    0.047945
Sex_male      0.047352
dtype: float64